In [1]:
import requests
import pandas as pd
import os
import re
from tqdm import tqdm #barra de progreso

session = requests.Session()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) #para ver toda la tabla, en nuestro caso son solo 30 se supone que no tenemos que tener problema

API_KEY = '2c6d07b262d82febbd27cf5327a36d55' 

ARTISTAS = [
    "La Fuga",
    "Fito y fitipaldis",
    "Billie Eilish",
    "Love of Lesbian",
    "Estopa",
    "Mägo de Oz",
    "Mr. Kilombo",
    "Rozalén",
    "Taburete",
    "Maná",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Enrique Bunbury",
    "Marea",
    "Joaquín Sabina",
    "Rosalía",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

ARTISTAS = [a.strip() for a in ARTISTAS]

def limpiar_bio(texto):
    """Elimina etiquetas HTML como <a href=...>Read more</a>"""
    return re.sub(r'<[^>]+>', '', texto).strip()

#Funcion para llamar a la API
def obtener_info_artista(nombre):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getinfo",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect" : 1,  
        "lang": "es"
     }
    
    response = session.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    return response.json()

#Función para obtener los top tracks del artista con sus estadísticas
def obtener_top_tracks(nombre, limite=50):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getTopTracks",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect": 1,
        "limit": limite #límite de canciones por artista
    }
    response = session.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    data = response.json()
    return [
        {
            "track": t["name"],
            "playcount": int(t["playcount"]),
            "listeners": int(t["listeners"])
        }
        for t in data.get("toptracks", {}).get("track", []) #devuelve nombre, playcount y listeners de cada canción
    ]

def procesar_artista(nombre):
    try:
        data = obtener_info_artista(nombre)
        artista = data["artist"]
        
        bio = limpiar_bio(artista["bio"]["content"])
        listeners = int(artista["stats"]["listeners"])
        playcount = int(artista["stats"]["playcount"])
        similares = [a["name"] for a in artista["similar"]["artist"]]
        tracks = obtener_top_tracks(nombre) #llamamos a la función de top tracks
        
        return {
            "artista": nombre,
            "biografia": bio,
            "listeners": listeners,
            "playcount": playcount,
            "similares": ", ".join(similares),
            "tracks": tracks #lista de diccionarios con track, playcount y listeners
        }
    
    except KeyError as e:
        print(f"[KeyError] '{nombre}': clave no encontrada {e}")  #except especifico que muestra qué clave faltó
        return None
    except session.RequestException as e:
        print(f"[RequestError] '{nombre}': {e}")
        return None

In [2]:
resultados = []
bar = tqdm(ARTISTAS, desc="Iniciando...")
 
for artista in bar:
    bar.set_description(f"Procesando: {artista}")
    info = procesar_artista(artista)
    if info:
        resultados.append(info)
 
print(f"\n✅ {len(resultados)}/{len(ARTISTAS)} artistas obtenidos correctamente")

session.close()

Procesando: Residente: 100%|██████████| 30/30 [00:11<00:00,  2.69it/s]       


✅ 30/30 artistas obtenidos correctamente


In [3]:
#Crear DataFrame principal con info de artistas
df_lastfm = pd.DataFrame(resultados).drop(columns=["tracks"]) #quitamos tracks para el df principal

# ── MEJORA: exportar a CSV para no tener que volver a llamar a la API ──
df_lastfm.to_csv("lastfm_artistas.csv", index=False, encoding="utf-8-sig")
print("✅ Datos guardados en lastfm_artistas.csv")

#Crear DataFrame separado para tracks con sus estadísticas
df_tracks = pd.DataFrame([
    {"artista": r["artista"], **t}
    for r in resultados
    for t in r["tracks"]
])
df_tracks.to_csv("lastfm_tracks.csv", index=False, encoding="utf-8-sig")
print("✅ Tracks guardados en lastfm_tracks.csv")

df_lastfm.head() #con esto veo 5

✅ Datos guardados en lastfm_artistas.csv
✅ Tracks guardados en lastfm_tracks.csv


,artista,biografia,listeners,playcount,similares
0,La Fuga,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692,2804350,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
1,Fito y fitipaldis,"Adolfo Cabrales Mato, ""Fito"" (6 de octubre de ...",126372,3780711,"El Canto del Loco, Extremoduro, Marea, La Fuga..."
2,Billie Eilish,Billie Eilish Pirate Baird O'Connell​ (Los Áng...,4248142,841110926,"FINNEAS, Olivia Rodrigo, Melanie Martinez, Lan..."
3,Love of Lesbian,Love of Lesbian es un grupo de pop independien...,176699,11454248,"Lori Meyers, Izal, Viva Suecia, Vetusta Morla,..."
4,Estopa,Estopa es un grupo de pop/rumba/flamenco funda...,309521,8984044,"Melendi, El Canto del Loco, Extremoduro, Perez..."


In [8]:
#para probar artista
procesar_artista('fito & fitipaldis')

{'artista': 'fito & fitipaldis',
 'biografia': 'Adolfo Cabrales Mato, "Fito" (6 de octubre de 1966 en Bilbao, España) decidió fundar un grupo paralelo en 1998 con la intención de publicar temas un tanto alejados del patrón del rock and roll habitual de su anterior banda Platero y Tú, mucho más dura. Se trataba de dar salida a historias personales pasando por el blues, el rock, el soul e incluso el swing.\n\nDicho proyecto arrancó ese mismo año con la publicación de A puerta cerrada, un disco principalmente acústico con un sonido muy sencillo y cercano. Este álbum debut incluía una versión de Enrique Urquijo, Quiero beber hasta perder el control. También hubo un tema, Trozos de cristal, en el que colaboró Robe Iniesta, de Extremoduro. Rojitas las orejas, fue más tarde grabado por otro proyecto en el que Fito participó, Extrechinato y Tú; aunque esta vez con un sonido más rockero, uniendo los versos de Manolo Chinato a la letra del tema de Fito sobre una base musical más potente.\n\nEn 2

In [4]:
df_deezer = pd.read_csv("deezer_artists.csv") #carg el CSV de Deezer y lo convierte en una tabla

df_final = df_deezer.merge(df_lastfm, left_on="artist_name", right_on="artista", how="left") #Une las dos tablas usando el nombre del artista como union (es como un JOIN de SQL)
df_final = df_final.drop(columns=["artista"])  # eliminamos columna duplicada (artist_name en deezer y artista de last.fm, deja solo una)

df_final.to_csv("df_final.csv", index=False, encoding="utf-8-sig")
print("✅ Merge guardado en df_final.csv")
df_final.head()

✅ Merge guardado en df_final.csv


,artist_id,artist_name,album_title,track_title,type,year,genre,genre_id,biografia,listeners,playcount,similares
0,10786,La Fuga,Justo Después Del Silencio,Este Blues,track,2025,Rock,152,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692.0,2804350.0,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
1,10786,La Fuga,Justo Después Del Silencio,Cuántos Años,track,2025,Rock,152,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692.0,2804350.0,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
2,10786,La Fuga,Justo Después Del Silencio,Horas Infinitas,track,2025,Rock,152,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692.0,2804350.0,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
3,10786,La Fuga,Justo Después Del Silencio,A Ratos,track,2025,Rock,152,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692.0,2804350.0,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
4,10786,La Fuga,Justo Después Del Silencio,Flores de Mentira,track,2025,Rock,152,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82692.0,2804350.0,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
